In [ ]:
!pip install -q faster-whisper fastapi uvicorn python-multipart pyngrok torch torchaudio

In [ ]:
# resemble-enhance (pip) trava as dependências em versões exatas de 2023
# (torch==2.1.1, numpy==1.26.2, deepspeed==0.12.4 etc.) que conflitam com o
# que o Colab já tem instalado. Instalamos sem resolver dependências
# (--no-deps) e completamos na mão com versões atuais — funciona porque os
# pins são só desatualização do pacote (que não teve outra release desde
# 2023), não incompatibilidade real de API.
#
# Se essa célula falhar, o resto do notebook (pyngrok, uvicorn, /transcribe)
# continua funcionando — só o endpoint /enhance ficaria indisponível até
# você resolver o erro daqui.
!pip install -q --no-deps resemble-enhance
!pip install -q deepspeed omegaconf celluloid librosa matplotlib pandas ptflops resampy scipy soundfile tabulate torchvision

In [ ]:
%%writefile app.py
import os
import tempfile
import time
from contextlib import suppress

import soundfile as sf
import torch
from fastapi import BackgroundTasks, Depends, FastAPI, File, Form, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from fastapi.security import HTTPAuthorizationCredentials, HTTPBearer
from faster_whisper import WhisperModel

TRANSCRIBE_MODEL_NAME = "large-v3-turbo"
ENHANCE_MODEL_NAME = "resemble-enhance"
MAX_UPLOAD_MB = 500

API_TOKEN = os.environ["API_TOKEN"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"


def _remove_file(path: str) -> None:
    with suppress(OSError):
        os.remove(path)

app = FastAPI(
    title="Speech API",
    version="1.0.0"
)

# ==========================
# CORS
# ==========================

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ==========================
# AUTH
# ==========================

bearer_scheme = HTTPBearer()


def require_api_token(
    credentials: HTTPAuthorizationCredentials = Depends(bearer_scheme),
) -> None:
    if credentials.credentials != API_TOKEN:
        raise HTTPException(status_code=401, detail="Invalid or missing API token")


print(f"Carregando modelo de transcrição {TRANSCRIBE_MODEL_NAME} ({DEVICE}/{COMPUTE_TYPE})...")

transcribe_model = WhisperModel(
    TRANSCRIBE_MODEL_NAME,
    device=DEVICE,
    compute_type=COMPUTE_TYPE,
)

print("Modelo de transcrição carregado!")

# O modelo de enhance é importado sob demanda dentro do endpoint /enhance
# (não aqui no startup): assim, /transcribe e /enhance ficam independentes
# entre si — se o resemble-enhance falhar ao carregar/baixar pesos, isso
# não derruba a API de transcrição, e vice-versa.

# ==========================
# ROOT
# ==========================

@app.get("/")
def root():
    return {
        "name": "Speech API",
        "status": "online",
        "version": "1.0.0"
    }

# ==========================
# HEALTH
# ==========================

@app.get("/api/health")
def health():
    return {
        "status": "ok",
        "device": DEVICE,
        "compute_type": COMPUTE_TYPE,
        "version": "1.0.0",
        "endpoints": {
            "transcribe": {"model": TRANSCRIBE_MODEL_NAME, "loaded": True},
            "enhance": {"model": ENHANCE_MODEL_NAME, "loaded": "lazy"}
        }
    }

# ==========================
# TRANSCRIBE
# ==========================

@app.post("/transcribe", dependencies=[Depends(require_api_token)])
def transcribe(
    file: UploadFile = File(...),
    language: str = Form("pt"),
    vad: bool = Form(True),
    word_timestamps: bool = Form(True)
):
    start = time.time()

    suffix = os.path.splitext(file.filename)[1]
    content = file.file.read()

    max_bytes = MAX_UPLOAD_MB * 1024 * 1024
    if len(content) > max_bytes:
        raise HTTPException(
            status_code=413,
            detail=f"File too large (max {MAX_UPLOAD_MB}MB)",
        )

    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(content)
        path = tmp.name

    try:
        segments, info = transcribe_model.transcribe(
            path,
            language=None if language in ("", "auto") else language,
            vad_filter=vad,
            word_timestamps=word_timestamps
        )

        result_segments = []
        full_text = []

        words_count = 0

        duration = 0

        for seg in segments:

            duration = seg.end

            full_text.append(seg.text)

            words = []

            if seg.words:

                for w in seg.words:

                    words_count += 1

                    words.append({
                        "word": w.word,
                        "start": w.start,
                        "end": w.end
                    })

            result_segments.append({

                "start": seg.start,
                "end": seg.end,
                "text": seg.text,
                "words": words

            })
    except Exception as exc:
        raise HTTPException(
            status_code=422, detail=f"Transcription failed: {exc}"
        ) from exc
    finally:
        with suppress(OSError):
            os.remove(path)

    return {

        "success": True,

        "language": info.language,

        "duration": duration,

        "processing_time": round(time.time() - start, 2),

        "model": TRANSCRIBE_MODEL_NAME,

        "segments": result_segments,

        "segments_count": len(result_segments),

        "words_count": words_count,

        "text": "".join(full_text)

    }

# ==========================
# ENHANCE
# ==========================

@app.post("/enhance", dependencies=[Depends(require_api_token)])
def enhance(
    file: UploadFile = File(...),
    denoise_only: bool = Form(False),
    nfe: int = Form(64),
    lambd: float = Form(0.9),
    tau: float = Form(0.5),
    solver: str = Form("midpoint"),
):
    import numpy as np
    import scipy.optimize
    from resemble_enhance.enhancer.inference import denoise as resemble_denoise
    from resemble_enhance.enhancer.inference import enhance as resemble_enhance_fn
    from resemble_enhance.enhancer.lcfm import cfm as _resemble_cfm

    # resemble-enhance quebra em numpy>=2 porque faz float() de um array
    # numpy de 1 elemento (proibido desde o NEP 51). Patch mínimo pra extrair
    # o escalar de forma segura em qualquer versão de numpy/scipy.
    def _fixed_exponential_decay_mapping(t, n=4):
        def h(t, a):
            return (a**t - 1) / (a - 1)

        result = scipy.optimize.fsolve(lambda a: h(1 / n, a) - 0.5, x0=0)
        a = float(np.asarray(result).reshape(-1)[0])
        return h(t, a=a)

    _resemble_cfm.Solver.exponential_decay_mapping = staticmethod(
        _fixed_exponential_decay_mapping
    )

    start = time.time()

    suffix = os.path.splitext(file.filename)[1]
    content = file.file.read()

    max_bytes = MAX_UPLOAD_MB * 1024 * 1024
    if len(content) > max_bytes:
        raise HTTPException(
            status_code=413,
            detail=f"File too large (max {MAX_UPLOAD_MB}MB)",
        )

    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(content)
        input_path = tmp.name

    output_path = f"{input_path}_enhanced.flac"

    try:
        data, sr = sf.read(input_path, dtype="float32")
        if data.ndim > 1:
            data = data.mean(axis=1)
        dwav = torch.from_numpy(data)

        if denoise_only:
            enhanced_wav, new_sr = resemble_denoise(dwav, sr, DEVICE)
        else:
            enhanced_wav, new_sr = resemble_enhance_fn(
                dwav, sr, DEVICE, nfe=nfe, lambd=lambd, tau=tau, solver=solver
            )

        sf.write(output_path, enhanced_wav.cpu().numpy(), new_sr, format="FLAC")
    except Exception as exc:
        with suppress(OSError):
            os.remove(output_path)
        raise HTTPException(
            status_code=422, detail=f"Enhance failed: {exc}"
        ) from exc
    finally:
        with suppress(OSError):
            os.remove(input_path)

    cleanup = BackgroundTasks()
    cleanup.add_task(_remove_file, output_path)

    return FileResponse(
        output_path,
        media_type="audio/flac",
        filename="enhanced.flac",
        background=cleanup,
        headers={
            "X-Model": ENHANCE_MODEL_NAME,
            "X-Processing-Time": str(round(time.time() - start, 2)),
        },
    )


In [ ]:
import os

from pyngrok import ngrok
from google.colab import userdata

ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))

# Crie um secret "API_TOKEN" no Colab (ícone de chave na barra lateral) com
# um valor aleatório seu — ele protege os endpoints /transcribe e /enhance
# de uso por qualquer pessoa que descubra a URL pública do ngrok.
os.environ['API_TOKEN'] = userdata.get('API_TOKEN')

public_url = ngrok.connect(8000)

print(public_url)
print('Header em ambos os endpoints: Authorization: Bearer <seu API_TOKEN>')
print('POST /transcribe -> transcrição (faster-whisper)')
print('POST /enhance    -> speech enhancement (resemble-enhance), independente do /transcribe')

In [ ]:
import subprocess
import time

log_file = open('uvicorn.log', 'w')
uvicorn_process = subprocess.Popen(
    ['uvicorn', 'app:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

time.sleep(3)
print(f'Uvicorn iniciado (PID {uvicorn_process.pid}). Logs em uvicorn.log')
print(f'Ainda rodando: {uvicorn_process.poll() is None}')
# Pra ver os logs a qualquer momento: !tail -n 50 uvicorn.log
# Pra derrubar o servidor: uvicorn_process.terminate()